# Cardio-X: Exploratory Data Analysis & Feature Engineering
## Project: An Intelligent ML-Driven System for Cholesterol Risk Detection, Prevention & Personalized Lifestyle Advisory

### Academic Overview
This notebook provides a systematic, reproducible exploratory investigation of the Framingham longitudinal cardiovascular cohort.

**Key Objectives:**
1. Audit the dataset structure, missingness patterns, and continuous biomarker distributions.
2. Characterize the longitudinal multi-wave examination design (`PERIOD` 1, 2, and 3).
3. Differentiate between full-study long-term surveillance (~24 years) and prospective 10-year risk horizons.
4. Safely compute evidence-based clinical hemodynamic and lipid features (`PULSE_PRESSURE`, `MAP`, `NON_HDL_CHOL`, `CHOL_HDL_RATIO`).
5. Evaluate paired trajectories for participants attending all three examination cycles ($N=3,206$).

> **Academic Note:** Descriptive statistical correlation does not establish clinical causation. No models are trained in this phase.

In [1]:
import hashlib
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Configure aesthetic plotting parameters
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

# Verify data integrity
DATA_PATH = Path("../data/FraminghamDataset.csv")
assert DATA_PATH.is_file(), f"Dataset not found at {DATA_PATH}"
with open(DATA_PATH, "rb") as f:
    sha256_hash = hashlib.sha256(f.read()).hexdigest()
print(f"Dataset verified: {DATA_PATH.resolve()}")
print(f"SHA-256 Digest : {sha256_hash}")

df_raw = pd.read_csv(DATA_PATH)
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

Dataset verified: /Users/gauravkumar/Desktop/CARDIO-X/data/FraminghamDataset.csv
SHA-256 Digest : e073ce4c60d351f2015c500b5bdf89c3a99ab3dfd0ba525e31a743742d649131
Shape: 11,627 rows x 39 columns


### 1. Clinical Feature Engineering
We engineer four established hemodynamic and lipid markers without in-place mutation of the raw data:
- **Pulse Pressure (PP):** $\text{PP} = \text{SYSBP} - \text{DIABP}$
- **Mean Arterial Pressure (MAP):** $\text{MAP} = \text{DIABP} + \frac{1}{3}(\text{SYSBP} - \text{DIABP})$
- **Non-HDL Cholesterol:** $\text{Non-HDL} = \text{TOTCHOL} - \text{HDLC}$ (only where both are present)
- **Cholesterol-to-HDL Ratio:** $\text{Ratio} = \frac{\text{TOTCHOL}}{\text{HDLC}}$ (safe division guarded against zero/missing denominators)

In [2]:
import sys
sys.path.append("..")
from src.eda import calculate_engineered_features, compute_distribution_metrics

df = calculate_engineered_features(df_raw)
engineered_cols = ["PULSE_PRESSURE", "MAP", "NON_HDL_CHOL", "CHOL_HDL_RATIO"]
df[engineered_cols].describe().T[["count", "mean", "std", "min", "50%", "max"]]

,count,mean,std,min,50%,max
PULSE_PRESSURE,11627.0,53.286359,16.654655,15.000000,50.000000,170.000000
MAP,11627.0,100.799877,14.221777,62.500000,99.000000,188.333333
NON_HDL_CHOL,3027.0,187.338289,44.451923,60.000000,183.000000,565.000000
CHOL_HDL_RATIO,3027.0,5.258144,2.054787,1.550265,4.857143,33.600000


### 2. Missingness Profile Across Examination Cycles
Examination of why HDL and LDL exhibit ~74% missingness across the dataset.

In [3]:
features_to_check = ["TOTCHOL", "HDLC", "LDLC", "GLUCOSE", "BPMEDS", "BMI", "CIGPDAY"]
pct_missing_by_period = df.groupby("PERIOD")[features_to_check].apply(lambda g: (g.isnull().mean() * 100).round(1))

plt.figure(figsize=(10, 4.5))
sns.heatmap(pct_missing_by_period, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={"label": "% Missing"})
plt.title("Missingness Percentage by Examination Period (Structural Protocol Audit)", fontweight="bold")
plt.xlabel("Clinical Feature")
plt.ylabel("Examination Period")
plt.tight_layout()
plt.show()

/var/folders/f7/yx07bbts1_9b7hx58vtjts4h0000gn/T/ipykernel_40516/3834908751.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3. Correlation Analysis
We compute two separate correlation structures:
1. **Baseline Cohort Correlation Matrix ($N=4,434$):** Evaluates baseline predictors and outcomes without longitudinal confounding.
2. **Period 3 Complete Lipid Panel Matrix ($N=3,026$):** Evaluates fractionated lipid subfractions (`HDLC`, `LDLC`, `NON_HDL_CHOL`, `CHOL_HDL_RATIO`).

In [4]:
base_df = df[df["PERIOD"] == 1]
corr_cols = ["AGE", "SYSBP", "DIABP", "PULSE_PRESSURE", "MAP", "BMI", "GLUCOSE", "CIGPDAY", "TOTCHOL", "CVD", "ANYCHD"]
corr = base_df[corr_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-0.4, vmax=1.0, annot=True, fmt=".2f", square=True, linewidths=0.5)
plt.title("Correlation Matrix: Baseline Predictors & Endpoints (Period 1, N=4,434)", fontweight="bold")
plt.tight_layout()
plt.show()

/var/folders/f7/yx07bbts1_9b7hx58vtjts4h0000gn/T/ipykernel_40516/2086225692.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4. Paired Longitudinal Trajectories (N=3,206 Participants Attending All 3 Cycles)
By tracking the identical 3,206 participants across all 3 examination waves, we avoid survivor and dropout bias.

In [5]:
from src.eda import compute_paired_longitudinal_trends
n_paired, long_df = compute_paired_longitudinal_trends(df)
print(f"Paired Longitudinal Cohort Size: {n_paired:,} participants")
long_df

Paired Longitudinal Cohort Size: 3,206 participants


,Feature,Period,Paired_Participants,Valid_Observations,Mean,Std,Median
0,AGE,1,3206,3206,48.71,8.32,48.00
1,AGE,2,3206,3206,54.67,8.30,54.00
2,AGE,3,3206,3206,60.64,8.30,60.00
3,SYSBP,1,3206,3206,130.13,20.07,127.00
4,SYSBP,2,3206,3206,135.67,21.59,132.00
5,SYSBP,3,3206,3206,140.18,22.92,137.00
6,DIABP,1,3206,3206,82.18,11.29,81.00
7,DIABP,2,3206,3206,83.85,11.07,83.00
8,DIABP,3,3206,3206,81.79,11.25,80.00
9,TOTCHOL,1,3206,3174,235.40,44.02,232.00


### 5. Surveillance Horizon Evaluation: Full Study (~24 Years) vs. 10-Year Prospective Window
Clarifying the outcome observation horizons.

In [6]:
from src.eda import analyze_outcome_surveillance_windows
surv_df = analyze_outcome_surveillance_windows(df)
surv_df

,Endpoint,Description,Baseline_Participants,Full_Surveillance_Events,Full_Surveillance_Rate_Pct,Ten_Year_Window_Events,Ten_Year_Window_Rate_Pct,Max_Follow_Up_Years
0,CVD,Cardiovascular Disease,4434,1157,26.09,530,11.95,24.0
1,ANYCHD,Any Coronary Heart Disease,4434,1240,27.97,573,12.92,24.0
2,HYPERTEN,Hypertension,4434,3252,73.34,2525,56.95,24.0
3,STROKE,Stroke,4434,415,9.36,143,3.23,24.0
4,DEATH,All-Cause Mortality,4434,1550,34.96,429,9.68,24.0


### 6. Conclusions & Academic Recommendations for Phase 4
1. **Cohort Selection:** Baseline `PERIOD == 1` ($N=4,434$) is recommended for primary cardiovascular risk modeling to prevent longitudinal leakage.
2. **Horizon Clarification:** Models predicting 10-year risk must use the windowed target (`TIMECVD <= 3652.5`), whereas 24-year models represent lifetime/extended cohort surveillance.
3. **Lipid Modeling:** Complete lipid profiles (`HDLC`, `LDLC`) can be modeled in a dedicated secondary model on Period 3 ($N=3,026$) or through staged risk assessment.